## Импорты

In [148]:
import importlib
import json
import os
import pickle
import random
import sys
import warnings

import matplotlib.pyplot as plt


import numpy as np
import tensorflow as tf

from tensorflow.keras import Model, layers, optimizers, losses
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.optimizers import Adam

from typing import List, Dict, Any, Tuple
from music21 import chord, note, instrument, environment

import torch
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")

# Всё, что выводит CUDA / XLA, будет записано в файл
sys.stderr = open("xla_warnings.log", "w")



sys.path.append('src')


# в режиме редактирования сбрасуем кэшь иначе не видно изменений
for module in [
    "utils.music21_utils",
    "utils.data_utils",
    "utils.generate_utils",
    #"utils.train_utils",
    "utils.dataset_utils",
    "utils.tokenizer_utils",
    "utils.model_utils",
    "utils.train_utils",
    "utils.__init__",
    "models.__init__",
    "src.constants",
    "src.stages.__init__"
    
]:
    imported_module = importlib.import_module(module)
    importlib.reload(imported_module)

from src.constants import (
    PITCH_VOCAB, LR_PRETRAIN, DEVICE, PRETRAIN_EPOCHS,
    FINETUNE_EPOCHS, CACHE_DIR
)

from src.stages import (
    stage_train_melody
)

from utils.__init__ import (
    denormalize_sequence_global,
    get_all_midis,
    Music21Tokenizer,
    MusicTransformerED,
    MIDIDataset,
    collate_chunks,
    detect_instruments_in_midi,
    extract_sample,
    EarlyStopping,
    train_epoch,
    generate_autoregressive,
    save_melody_midi
)

from models.__init__ import (
    DecoderOnlyMusicTransformer,
    MelodyLoss
)


# настраиваем окружение в соответствии с документацией
us = environment.UserSettings()
environment.set('midiPath', '/usr/bin/timidity')
us['musescoreDirectPNGPath'] = '/usr/bin/mscore3'
us['directoryScratch'] = '/tmp'

Device: cuda


## Загрузка midi файлов

In [3]:
# Считаем только Chopin
chopin_midis = get_all_midis("../dataset/Chopin")

## Собираем все инструменты

In [4]:
# instruments = []
# for score in all_pretrain_midis:
#    instruments.extend(detect_instruments_in_midi(score))
# global_instruments = set(instruments)

## Собираем сэмплы со всеми инструментами если нет то пропуск

In [134]:
#samples_pkl = 'cache/samples.pkl'
#with open(samples_pkl, "rb") as f:
#    samples = pickle.load(f)

In [7]:
## Инициализируем токенизатор

In [126]:
tokenizer = Music21Tokenizer()

## Готовим датасет

In [138]:
## фильтруем где есть piano
# filtered = [s for s in samples if len(s.get(target_instrument, []))>0]

In [76]:
## 2. Глобальная статистика

In [149]:
# plt.figure(figsize=(18, 3), facecolor='#97BACB')
# bins = np.arange(0,(max(Recurrence)), 1000)
# plt.hist(Recurrence, bins=bins, color='#97BACB')
# plt.axvline(x=100, color='#DBACC1') # отсечка по встречаемости в 100 раз
# plt.title('Распределение нот в корпусе')
# plt.xlabel('Частота встречаемости ноты в корпусе')
# plt.ylabel('Число нот')
# plt.show()

# print("Длина корпуса после исключения редких нот:", len(all_notes))

### Составляем глобальную статику


stats_json = "cache/stats.json"
with open(stats_json, "r") as f:
    stats = json.load(f)
print("Global stats:", stats)

pitch_offset = int(stats["pitch_min"])  # обязательный

prepared_pkl = CACHE_DIR / "prepared_items.pkl"

with open(prepared_pkl, "rb") as f:
    prepared = pickle.load(f)
    items = prepared["items"]

Global stats: {'pitch_min': 21, 'pitch_max': 107, 'step_max': 1567.59375, 'dur_max': 3.0}


In [146]:
## готовим датасет для предобученной модели
# -------------------------
# 3) DataLoader
# -------------------------
# dataset = MIDIDataset(filtered, tokenizer, stats, pitch_offset,
#                      target_instrument='Piano',
#                      input_instruments=[], L_enc=100, L_dec=32)

# train_loader = DataLoader(dataset,
#                    batch_size=16,   # batch_size по "сэмплам" = количество items из dataset
#                    shuffle=True,
#                    collate_fn=collate_chunks,
#                    drop_last=True)

In [131]:
## подготавливем данные для Fine tuning
### 3. Подготовка данных
samples_chopin = []
for score in chopin_midis:
    sample = extract_sample(score, global_instruments)
    samples_chopin.append(sample)
    
filtered_chopin = [s for s in samples_chopin if len(s.get(target_instrument, []))>0]

# -------------------------
# 3) DataLoader
# -------------------------
chopin_dataset = MIDIDataset(filtered_chopin, tokenizer, stats, pitch_offset,
                      target_instrument='Piano',
                      input_instruments=[], L_enc=100, L_dec=32)

ft_loader = DataLoader(chopin_dataset,
                    batch_size=16,   # batch_size по "сэмплам" = количество items из dataset
                    shuffle=True,
                    collate_fn=collate_chunks,
                    drop_last=True)

## Модель

In [140]:
# Model init
#model = DecoderOnlyMusicTransformer(pitch_vocab=PITCH_VOCAB, d_model=256, n_heads=8, n_layers=6).to(DEVICE)
#criterion = MelodyLoss()
#optimizer = torch.optim.Adam(model.parameters(), lr=LR_PRETRAIN)

## Обучение

### Обучение предобученной модели

In [150]:
#early_stopping = EarlyStopping(patience=5, min_delta=1e-4)
#pretrain_ckpt = "decoder_pretrain.pt"

# Pretrain loop
#print("Start pretraining...")

# for epoch in range(PRETRAIN_EPOCHS):

#    loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
#    print(f"[Pretrain] Epoch {epoch+1}/{PRETRAIN_EPOCHS} loss={loss:.4f}")

    # Early stopping
#    early_stopping.step(loss)

    # Сохраняем лучшую модель
#    if loss == early_stopping.best_loss:
#        torch.save(model.state_dict(), pretrain_ckpt)
#        print(f"New best model saved at epoch {epoch+1}")

#    if early_stopping.should_stop:
#        print(f"Early stopping triggered at epoch {epoch+1}")
#        break

#print("Final best model saved as:", pretrain_ckpt)


stage_train_melody(items)

### FineTuning

In [84]:
# load pretrain weights
model.load_state_dict(torch.load(pretrain_ckpt, map_location=DEVICE))
# optimizer small LR
optimizer = torch.optim.Adam(model.parameters(), lr=LR_FINETUNE)

early_stopping = EarlyStopping(patience=5, min_delta=1e-4)

ft_ckpt = "decoder_finetune_chopin.pt"

# Fine-tuning loop
print("Start fine-tuning on Chopin...")

for epoch in range(FINETUNE_EPOCHS):

    loss = train_epoch(model, ft_loader, optimizer, criterion, DEVICE)
    print(f"[FineTune] Epoch {epoch+1}/{FINETUNE_EPOCHS} loss={loss:.4f}")

    # Early stopping
    early_stopping.step(loss)

    # Сохраняем лучшую модель
    if loss == early_stopping.best_loss:
        torch.save(model.state_dict(), ft_ckpt)
        print(f"New best model saved at epoch {epoch+1}")

    if early_stopping.should_stop:
        print(f"Early stopping triggered at epoch {epoch+1}")
        break

print("Final best model saved as:", ft_ckpt)

Start fine-tuning on Chopin...
[FineTune] Epoch 1/50 loss=0.0088
New best model saved at epoch 1
[FineTune] Epoch 2/50 loss=0.0087
[FineTune] Epoch 3/50 loss=0.0085
New best model saved at epoch 3
[FineTune] Epoch 4/50 loss=0.0076
New best model saved at epoch 4
[FineTune] Epoch 5/50 loss=0.0080
[FineTune] Epoch 6/50 loss=0.0077
[FineTune] Epoch 7/50 loss=0.0074
New best model saved at epoch 7
[FineTune] Epoch 8/50 loss=0.0073
New best model saved at epoch 8
[FineTune] Epoch 9/50 loss=0.0073
[FineTune] Epoch 10/50 loss=0.0072
[FineTune] Epoch 11/50 loss=0.0075
[FineTune] Epoch 12/50 loss=0.0076
[FineTune] Epoch 13/50 loss=0.0073
Early stopping triggered at epoch 13
Final best model saved as: decoder_finetune_chopin.pt


## Генирация

### Генерация мелодии

In [90]:
# -------------------------
# Generation example
# -------------------------
# prepare seed: one note (C4 ~ 60) or sample from dataset
# seed pitch should be stored as label (pitch - pitch_offset)
seed_pitch = 60
seed_label = seed_pitch - pitch_offset
seed = np.array([[[seed_label, 0.0, 0.5]]], dtype=np.float32)  # normalized: step and dur are relative -> we'll normalize below
# normalize seed step/dur
seed[0,0,1] = seed[0,0,1] / float(stats['step_max'])
seed[0,0,2] = seed[0,0,2] / float(stats['dur_max'])

# load finetuned model if exists
if os.path.exists("decoder_finetune_chopin.pt"):
    model.load_state_dict(torch.load("decoder_finetune_chopin.pt", map_location=DEVICE))
elif os.path.exists("decoder_pretrain.pt"):
    model.load_state_dict(torch.load("decoder_pretrain.pt", map_location=DEVICE))
model.to(DEVICE)
model.eval()

DecoderOnlyMusicTransformer(
  (embed): EventEmbedding(
    (pitch_emb): Embedding(128, 256)
    (step_linear): Linear(in_features=1, out_features=256, bias=True)
    (dur_linear): Linear(in_features=1, out_features=256, bias=True)
    (proj): Linear(in_features=768, out_features=256, bias=True)
  )
  (decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (multihead_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNor

In [103]:
generated_norm = generate_autoregressive(model, seed, length=GEN_LENGTH, pitch_temp=0.9, cont_temp=0.02, device=DEVICE)
# generated_norm shape (1, L+1, 3) — first token seed
# Denormalize (to real pitch, step, dur)
generated_den = denormalize_sequence_global(generated_norm, stats)
# generated_den may be (1, T, 3)
if generated_den.ndim == 3 and generated_den.shape[0] == 1:
    gen_seq = generated_den[0, 1:, :]  # drop seed
else:
    gen_seq = generated_den[1:, :]

out_fp = "generated_melody.mid"
save_melody_midi(gen_seq, fp=out_fp, instr_name="Piano")
print("Saved generated melody:", out_fp)